# Bài 14: Mô hình ngôn ngữ lớn
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Ô 1 đến ô 6 là phần lõi, chỉ dùng Python và NumPy. Ô 7 là phần **mở rộng, kết quả tuỳ máy**. Kho chữ là 34 câu về lớp 10A3 do nhóm tác giả tự viết, dùng lại từ Bài 8 (Tập 1).
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Kho chữ cũ của Bài 8

In [ ]:
kho = [
    "cô hạnh dạy toán ở lớp mười a ba", "thầy nam dạy lý ở lớp mười a ba",
    "cô mai dạy văn ở lớp mười a ba", "cô hạnh rất hiền", "thầy nam rất vui tính",
    "cô mai rất nghiêm", "minh thích học toán", "lan thích học văn",
    "khoa thích học lý", "minh thích chơi bóng đá", "lan thích đọc sách",
    "khoa thích chơi cờ vua", "em thích học toán với cô hạnh",
    "em thích học văn với cô mai", "em thích học lý với thầy nam",
    "sáng thứ hai lớp em chào cờ", "chiều thứ năm lớp em đá bóng",
    "em đi học bằng xe đạp", "em đi học bằng xe buýt", "minh đi học bằng xe đạp",
    "lan đi học bằng xe buýt", "khoa đi học bằng xe máy điện",
    "lớp em có ba mươi lăm bạn", "lớp em học ở tầng hai",
    "thư viện trường em ở tầng một", "em đọc sách ở thư viện",
    "em ăn sáng ở căng tin", "căng tin trường em bán bánh mì",
    "mai lớp em kiểm tra toán", "thứ sáu lớp em kiểm tra văn",
    "cô hạnh chấm bài rất nhanh", "thầy nam cho nhiều bài tập",
    "cô mai kể chuyện rất hay", "em rất thích lớp mười a ba",
]
print("Số câu:", len(kho), "| số chữ cái:", sum(len(c.replace(" ", "")) for c in kho))

### Ô 2. Tự học cách cắt token (BPE)
Bắt đầu từ từng chữ cái. Mỗi lượt, tìm **cặp đứng cạnh nhau nhiều nhất** rồi ghép thành một token mới.

In [ ]:
from collections import Counter

def hoc_bpe(kho, so_luot):
    tan_so = Counter(t for cau in kho for t in cau.split())
    tach = {t: list(t) + ["·"] for t in tan_so}     # "·" đánh dấu cuối từ
    luat = []
    for _ in range(so_luot):
        cap = Counter()
        for t, f in tan_so.items():
            for a, b in zip(tach[t], tach[t][1:]):
                cap[(a, b)] += f
        (a, b), _ = cap.most_common(1)[0]
        luat.append((a, b))
        for t in tach:                               # ghép cặp đó ở mọi từ
            s, moi, i = tach[t], [], 0
            while i < len(s):
                if i + 1 < len(s) and (s[i], s[i + 1]) == (a, b):
                    moi.append(a + b); i += 2
                else:
                    moi.append(s[i]); i += 1
            tach[t] = moi
    return luat

luat_vi = hoc_bpe(kho, 60)
print("10 lượt ghép đầu:", [a + b for a, b in luat_vi[:10]])

### Ô 3. Cắt một câu thành token

In [ ]:
def cat_token(cau, luat):
    ket_qua = []
    for t in cau.split():
        s = list(t) + ["·"]
        for a, b in luat:                            # áp dụng luật theo đúng thứ tự đã học
            i, moi = 0, []
            while i < len(s):
                if i + 1 < len(s) and (s[i], s[i + 1]) == (a, b):
                    moi.append(a + b); i += 2
                else:
                    moi.append(s[i]); i += 1
            s = moi
        ket_qua += s
    return ket_qua

for cau in ["cô hạnh dạy toán", "thầy bình dạy sử"]:
    tk = cat_token(cau, luat_vi)
    print(f"{cau!r}: {len(cau.split())} từ -> {len(tk)} token")
    print("   ", tk)

## Chặng 2: Đọc
1. Ở ô 2, những token đầu tiên được ghép là gì? Vì sao lại là chúng?
2. Ở ô 3, vì sao “thầy bình dạy sử” bị cắt vụn hơn “cô hạnh dạy toán”?
3. Nếu chạy thêm thật nhiều lượt ghép, mỗi từ trong kho sẽ thành mấy token?

## Chặng 3: Sửa
### Ô 4. Bộ cắt token học từ tiếng Anh thì cắt tiếng Việt ra sao?

In [ ]:
kho_anh = ["my teacher teaches math in class ten", "the teacher is very kind",
           "she marks the tests very quickly", "we like to read books in the library",
           "we go to school by bus", "we go to school by bike",
           "my class has thirty five students", "we eat bread at the canteen",
           "we have a math test on friday", "he tells very good stories",
           "the library is on the first floor", "we play football on thursday"] * 3
luat_anh = hoc_bpe(kho_anh, 60)
cap_cau = [("she marks the tests very quickly", "cô chấm bài kiểm tra rất nhanh"),
           ("we go to school by bus", "chúng em đi học bằng xe buýt")]
for anh, viet in cap_cau:
    print(f"Tiếng Anh: {len(cat_token(anh, luat_anh)):2} token | "
          f"Tiếng Việt: {len(cat_token(viet, luat_anh)):2} token")

### Ô 5. Chạy lại ô đếm “ảo giác” của Bài 8, cho máy nhìn xa hơn
Bài 8 chỉ nhìn **1 từ** trước. Ở đây máy nhìn 1, 2 rồi 3 từ trước. Ta đếm hai thứ: số câu nói **sai** ai dạy môn gì, và tỉ lệ câu **chép nguyên** từ kho.

In [ ]:
import random
from collections import defaultdict

def hoc_ngram(kho, n):
    bang = defaultdict(Counter)
    for cau in kho:
        tu = ["<đầu>"] * n + cau.split() + ["<hết>"]
        for i in range(n, len(tu)):
            bang[tuple(tu[i - n:i])][tu[i]] += 1
    return bang

def sinh_cau(bang, n, toi_da=15):
    nho, cau = ["<đầu>"] * n, []
    while len(cau) < toi_da:
        dem = bang[tuple(nho)]
        tu = random.choices(list(dem), weights=list(dem.values()))[0]
        if tu == "<hết>":
            break
        cau.append(tu)
        nho = nho[1:] + [tu]
    return " ".join(cau)

SU_THAT = {"cô hạnh": "toán", "thầy nam": "lý", "cô mai": "văn"}
def dem_sai(ds):
    return sum(c.startswith(gv + " dạy ") and not c.startswith(gv + " dạy " + m)
               for c in ds for gv, m in SU_THAT.items())

for n in [1, 2, 3]:
    random.seed(1)
    bang = hoc_ngram(kho, n)
    ds = [sinh_cau(bang, n) for _ in range(1000)]
    chep = sum(c in kho for c in ds) / len(ds)
    print(f"Nhìn {n} từ: {dem_sai(ds):3} câu sai | {chep:4.0%} chép nguyên kho")

### Ô 6. Một mô hình ngôn ngữ thật có bao nhiêu tham số?
Công thức gần đúng cho mô hình kiểu GPT: mỗi lớp Transformer có khoảng 12 × d × d tham số (d là độ dài vectơ của mỗi token).

In [ ]:
def dem_tham_so(so_lop, d, so_token_tu_dien, do_dai_ngu_canh):
    transformer = 12 * so_lop * d * d
    bang_vecto = so_token_tu_dien * d + do_dai_ngu_canh * d
    return transformer + bang_vecto

gpt2_nho = dem_tham_so(so_lop=12, d=768, so_token_tu_dien=50_257, do_dai_ngu_canh=1_024)
print(f"GPT-2 bản nhỏ (2019): khoảng {gpt2_nho / 1e6:.0f} triệu tham số")
gap_lan = f"{gpt2_nho // 2410:,}".replace(",", ".")
print(f"Gấp {gap_lan} lần mạng nhận chữ số ở Bài 10 (2.410 tham số)")

## Mở rộng: kết quả tuỳ máy
### Ô 7. Đếm token bằng bộ cắt token thật
Cần Internet. `tiktoken` là thư viện cắt token mã nguồn mở của OpenAI.

In [ ]:
!pip install -q tiktoken
import tiktoken
bo_cat = tiktoken.get_encoding("cl100k_base")
for cau in ["We go to school by bus", "Chúng em đi học bằng xe buýt"]:
    print(len(bo_cat.encode(cau)), "token:", cau)

## Chặng 4: Tự làm
1. Ở ô 2, đổi 60 lượt ghép thành 10 rồi 200. Chạy lại ô 3. Số token thay đổi thế nào?
2. Ở ô 5, thêm `n = 4`. Tỉ lệ chép nguyên kho là bao nhiêu? Mô hình còn “sáng tạo” không?
3. Thêm vào ô 4 vài câu tiếng Việt vào `kho_anh`, chạy lại. Số token tiếng Việt có giảm không?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*